# 스트레스 점수 예측 V9

**Pooled Multi-seed Quantile ExtraTrees**

- 행 단위 파생변수 생성
- Train 데이터만 이용해 전처리 기준 학습
- ExtraTrees 1,200개 × 3개 시드
- 3,600개 트리 예측을 통합해 50.5% 분위수로 집계
- 최종 예측값을 0~1 범위로 제한

In [1]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

## 1. 설정

In [2]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
SEEDS = (42, 77, 2026)
N_ESTIMATORS = 1200
QUANTILE = 0.505

WEIGHTED_FEATURES = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

## 2. 데이터 불러오기 및 검증

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
if train[TARGET].isna().any():
    raise ValueError("Train 타깃에 결측값이 있습니다.")

print("Train:", train.shape)
print("Test:", test.shape)
train.head()

## 3. 행 단위 파생변수

In [4]:
def add_features(frame):
    # 다른 행이나 데이터셋의 통계를 사용하지 않습니다.
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)

    # max_features=1에서 중요 피처의 선택 확률을 높입니다.
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]

    return result


X_train = add_features(train.drop(columns=[TARGET, ID_COLUMN]))
X_test = add_features(test.drop(columns=[ID_COLUMN]))
y_train = train[TARGET]

print("파생변수 적용 후:", X_train.shape)

파생변수 적용 후: (3000, 38)


## 4. 누수 없는 전처리

In [5]:
categorical = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numerical = X_train.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    [
        (
            "categorical",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="missing",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(strategy="median", add_indicator=True),
            numerical,
        ),
    ]
)

# fit은 Train에만 수행하고 Test에는 transform만 수행합니다.
train_matrix = preprocessor.fit_transform(X_train)
test_matrix = preprocessor.transform(X_test)

print("전처리 완료:", train_matrix.shape, test_matrix.shape)

전처리 완료: (3000, 57) (3000, 57)


## 5. 5-Fold 교차검증

각 Fold의 Train 부분에서만 전처리 기준과 모델을 학습합니다.  
세 개 시드에서 생성한 3,600개 트리 예측을 통합하고 50.5% 분위수로 MAE를 계산합니다.

In [6]:
def predict_forest_quantile(model, matrix, quantile=QUANTILE):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.quantile(tree_predictions, quantile, axis=1)


cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

fold_mae = []
oof_v9 = np.zeros(len(X_train))

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_train),
    start=1,
):
    # Fold의 학습 데이터로만 전처리 기준을 학습합니다.
    fold_preprocessor = clone(preprocessor)
    fold_train_matrix = fold_preprocessor.fit_transform(
        X_train.iloc[train_idx]
    )
    fold_valid_matrix = fold_preprocessor.transform(
        X_train.iloc[valid_idx]
    )

    fold_tree_predictions = []

    for seed in SEEDS:
        fold_model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=1,
            max_features=1,
            random_state=seed,
            n_jobs=-1,
        )
        fold_model.fit(
            fold_train_matrix,
            y_train.iloc[train_idx],
        )
        tree_predictions = np.column_stack(
            [
                tree.predict(fold_valid_matrix)
                for tree in fold_model.estimators_
            ]
        )
        fold_tree_predictions.append(tree_predictions)

    pooled_tree_predictions = np.concatenate(
        fold_tree_predictions,
        axis=1,
    )
    valid_prediction = np.quantile(
        pooled_tree_predictions,
        QUANTILE,
        axis=1,
    )
    valid_prediction = np.clip(valid_prediction, 0.0, 1.0)
    oof_v9[valid_idx] = valid_prediction

    score = mean_absolute_error(
        y_train.iloc[valid_idx],
        valid_prediction,
    )
    fold_mae.append(score)
    print(f"Fold {fold} MAE: {score:.6f}")

fold_mae = np.array(fold_mae)
v9_cv_mae = mean_absolute_error(y_train, oof_v9)

print("V9 Fold MAE:", np.round(fold_mae, 6))
print("V9 Mean MAE:", f"{v9_cv_mae:.6f}")
print("V9 MAE Std:", f"{fold_mae.std():.6f}")

V8_CV_MAE = 0.148826
print("V8 Mean MAE:", f"{V8_CV_MAE:.6f}")
print("V9 vs V8:", f"{v9_cv_mae - V8_CV_MAE:+.6f}")

Fold 1 MAE: 0.135675
Fold 2 MAE: 0.151034
Fold 3 MAE: 0.148525
Fold 4 MAE: 0.158925
Fold 5 MAE: 0.149267
V9 Fold MAE: [0.135675 0.151034 0.148525 0.158925 0.149267]
V9 Mean MAE: 0.148685
V9 MAE Std: 0.007483
V8 Mean MAE: 0.148826
V9 vs V8: -0.000141


## 6. 전체 Train 학습 및 통합 트리 예측

In [7]:
def predict_forest_quantile(model, matrix, quantile=QUANTILE):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.quantile(tree_predictions, quantile, axis=1)


tree_predictions_by_seed = []

for seed in SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(train_matrix, y_train)
    tree_predictions = np.column_stack(
        [
            tree.predict(test_matrix)
            for tree in model.estimators_
        ]
    )
    tree_predictions_by_seed.append(tree_predictions)
    print(f"Seed {seed}: prediction complete")

pooled_test_tree_predictions = np.concatenate(
    tree_predictions_by_seed,
    axis=1,
)

Seed 42: prediction complete
Seed 77: prediction complete
Seed 2026: prediction complete


## 7. 최종 예측값 확인

In [8]:
pred_v9 = np.quantile(
    pooled_test_tree_predictions,
    QUANTILE,
    axis=1,
)
pred_v9 = np.clip(pred_v9, 0.0, 1.0)

print("Prediction shape:", pred_v9.shape)
print("Prediction range:", pred_v9.min(), pred_v9.max())
print("Prediction mean:", pred_v9.mean())
print("첫 10개 예측값:")
print(np.round(pred_v9[:10], 6))


Prediction shape: (3000,)
Prediction range: 0.0 1.0
Prediction mean: 0.49971101666666673
첫 10개 예측값:
[0.58 0.97 0.19 0.44 0.53 0.46 0.78 0.47 0.53 0.88]


## 8. 제출 파일 생성 및 확인

In [9]:
submission_v9 = submission.copy()
submission_v9[ID_COLUMN] = test[ID_COLUMN]
submission_v9[TARGET] = pred_v9

output_path = OUTPUT_DIR / "submit_v9_pooled_quantile_forest.csv"
submission_v9.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path)
print("Rows:", len(submission_v9))
submission_v9.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v9_pooled_quantile_forest.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.58
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.44
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.53
9,TEST_0009,0.88


### 예측값 요약 통계

In [10]:
submission_v9[TARGET].describe().to_frame(name="stress_score")

,stress_score
count,3000.000000
mean,0.499711
std,0.203053
min,0.000000
25%,0.410000
50%,0.500000
75%,0.580000
max,1.000000
